# DeepDream

**Buku:** *Deep Learning dengan Python* — Bab 14, Deep Learning Generatif

Memaksimalkan aktivasi lapisan tengah InceptionV3 dengan *gradient ascent* pada gambar masukan, diproses dalam beberapa skala (oktaf).

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Notebook ini **membutuhkan backend TensorFlow** karena memakai `tf.GradientTape` untuk gradien terhadap masukan.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- `K.gradients` + `K.function` (graf statis TF1) → `tf.GradientTape` di dalam `tf.function`.
- `K.set_learning_phase(0)` tidak diperlukan lagi; model dipanggil dengan `training=False`.
- `scipy.misc.imsave` (dihapus dari SciPy) → `keras.utils.save_img`; pengubahan ukuran memakai `tf.image.resize`.
- Gambar contoh diunduh otomatis; aktivasi tepi gambar diabaikan agar tidak muncul artefak pinggir.

In [ ]:
import os
# Backend wajib TensorFlow untuk notebook ini
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
import tensorflow as tf
import numpy as np

## 1. Gambar dasar

In [ ]:
import matplotlib.pyplot as plt

jalur_gambar = keras.utils.get_file("langit.jpg", origin="https://i.imgur.com/aGBdQyK.jpg")
plt.imshow(keras.utils.load_img(jalur_gambar)); plt.axis("off"); plt.show()

## 2. Model dan kontribusi lapisan
Lapisan awal menghasilkan pola geometris; lapisan lebih dalam menghasilkan bentuk seperti mata atau bulu.

In [ ]:
from keras.applications import inception_v3

model = inception_v3.InceptionV3(weights="imagenet", include_top=False)
KONTRIBUSI = {"mixed2": 0.2, "mixed3": 3.0, "mixed4": 2.0, "mixed5": 1.5}   # sama dengan buku
ekstraktor = keras.Model(model.inputs,
                         {nama: model.get_layer(nama).output for nama in KONTRIBUSI})

def hitung_loss(gambar):
    fitur = ekstraktor(gambar, training=False)
    loss = tf.zeros(())
    for nama, bobot in KONTRIBUSI.items():
        a = fitur[nama]
        skala = tf.cast(tf.reduce_prod(tf.shape(a)), "float32")
        loss += bobot * tf.reduce_sum(tf.square(a[:, 2:-2, 2:-2, :])) / skala   # abaikan tepi
    return loss

## 3. *Gradient ascent*

In [ ]:
@tf.function
def langkah_naik(gambar, laju):
    with tf.GradientTape() as tape:
        tape.watch(gambar)
        loss = hitung_loss(gambar)
    grad = tape.gradient(loss, gambar)
    grad /= tf.maximum(tf.reduce_mean(tf.abs(grad)), 1e-6)    # normalisasi gradien
    return loss, gambar + laju * grad

def gradient_ascent(gambar, iterasi, laju, loss_maks=None):
    laju = tf.constant(laju)
    for i in range(iterasi):
        loss, gambar = langkah_naik(gambar, laju)
        if loss_maks is not None and loss > loss_maks:
            break
        print(f"  iterasi {i}: loss = {float(loss):.2f}")
    return gambar

## 4. Pra- dan pascapemrosesan

In [ ]:
def muat_gambar(jalur):
    arr = keras.utils.img_to_array(keras.utils.load_img(jalur))[None]
    return inception_v3.preprocess_input(arr)          # skala ke [-1, 1]

def kembalikan_gambar(x):
    x = np.array(x).reshape(x.shape[1], x.shape[2], 3)
    x = (x / 2.0 + 0.5) * 255.0
    return np.clip(x, 0, 255).astype("uint8")

## 5. Proses multi-oktaf
Setiap oktaf memperbesar gambar 1,4×, lalu detail yang hilang akibat pengecilan disuntikkan kembali.

In [ ]:
LAJU, OKTAF, SKALA_OKTAF = 0.01, 3, 1.4
ITERASI = 20          # per oktaf
LOSS_MAKS = 15.0

asli = muat_gambar(jalur_gambar)
bentuk_asli = asli.shape[1:3]
daftar_bentuk = [tuple(int(d / SKALA_OKTAF ** i) for d in bentuk_asli) for i in range(OKTAF)][::-1]

kecil_asli = tf.image.resize(asli, daftar_bentuk[0])
gambar = tf.identity(kecil_asli)
for i, bentuk in enumerate(daftar_bentuk):
    print(f"Oktaf {i} dengan bentuk {bentuk}")
    gambar = tf.image.resize(gambar, bentuk)
    gambar = gradient_ascent(gambar, ITERASI, LAJU, LOSS_MAKS)
    diperbesar = tf.image.resize(kecil_asli, bentuk)
    sama_ukuran = tf.image.resize(asli, bentuk)
    gambar += sama_ukuran - diperbesar                 # suntikkan kembali detail yang hilang
    kecil_asli = tf.image.resize(asli, bentuk)

hasil = kembalikan_gambar(gambar.numpy())
keras.utils.save_img("deepdream.png", hasil)
plt.figure(figsize=(10, 7)); plt.imshow(hasil); plt.axis("off"); plt.show()

**Eksperimen:** ganti `KONTRIBUSI` dengan lapisan lain (`mixed0` sampai `mixed10`, lihat `model.summary()`) dan amati perubahan polanya.